# 01. Define and build genomic contexts

Build site- and gene-level annotations of the sorghum genome for downstream enrichment and variance-partitioning analyses. Site contexts describe genomic position, coding consequences, regulatory features and population-genetic properties; gene contexts describe expression, co-expression networks, gene families and recombination.

# 01. 定义并构建基因组 contexts

构建高粱基因组的位点及基因级注释，用于后续富集与方差分解分析。位点 contexts 描述基因组位置、编码后果、调控特征和群体遗传属性；基因 contexts 描述表达、共表达网络、基因家族及重组率。

### Initialize the project

Set the project root and load the analysis environment. Input files are read from `data/source`, and generated tables are saved under `data/results`. Computational cells reuse existing results when inputs and parameters are unchanged. Context tables use HDF key `df`; variant-statistics tables are organized by chromosome.

### 初始化项目

设置项目根目录并载入分析环境。从 `data/source` 读取输入文件，将生成的表格保存到 `data/results`。输入及参数未变时，计算 cells 复用已有结果。Context 表采用 HDF key `df`；变异统计表按染色体组织。

In [4]:
import warnings
warnings.filterwarnings('ignore')

import os
import socket
import sys
import json
from datetime import datetime
from time import perf_counter
import pandas as pd
import yaml

%load_ext autoreload
%autoreload 2

project_dir = '/lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum'
with open(f'{project_dir}/config/project.yaml') as handle:
    project_config = yaml.safe_load(handle)
if project_dir != project_config['remote']['project_root']:
    raise ValueError('The explicit project root must match config/project.yaml.')
os.chdir(project_dir)
project_dir = os.getcwd()
data_dir = f'{project_dir}/data'
source_dir = f'{data_dir}/source'
results_dir = f'{data_dir}/results'
os.makedirs(results_dir, exist_ok=True)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from src.plot import set_plot_style
from src.contexts import save_context_table
from src.utils import have_file
set_plot_style()

print(f'Host: {socket.gethostname()}')
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.executable}')
print(f"Environment prefix: {os.environ.get('CONDA_PREFIX', '<not set>')}")

Host: rock01.local
Working directory: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum
Python: /lustre/home/dobby/Program/agent_program/micromamba-root/envs/dobby-base/bin/python
Environment prefix: /lustre/home/dobby/Program/agent_program/micromamba-root/envs/dobby-base


## 1. Define site-level contexts

## 1. 定义位点级 contexts

### 1.1 Prepare VCF inputs and quality control

The input SNP and indel VCFs have undergone genotype imputation and contain no missing genotypes.

Merge coordinate-sorted SNP and indel VCFs with identical sample order, keeping one record per position and preferring a SNP when both are present. Extract samples listed in the metadata `LIB` column and retain records with at least one ALT allele in the selected panel.

Genome-wide QC retains MAF ≥ 0.05 and heterozygosity ≤ 0.05. The CDS panel uses MAF ≥ 0.01 with the same heterozygosity limit; CDS intervals are taken from transcripts flagged `longest=1`.

### 1.1 准备 VCF 输入与质控

输入的 SNP 和 indel VCF 已完成基因型填补（imputation），基因型无缺失。

合并坐标排序且样本顺序相同的 SNP 和 indel VCF，每个位置保留一条记录，两者同时存在时优先保留 SNP。提取 metadata 的 `LIB` 列所列样本，保留所选群体中至少含一个 ALT 等位基因的记录。

全基因组质控保留 MAF ≥ 0.05、杂合率 ≤ 0.05。CDS 数据采用 MAF ≥ 0.01 及相同杂合率限制；CDS 区间取自标记为 `longest=1` 的转录本。

In [5]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Preparing or reusing VCF inputs')

import pandas as pd
from src.vcf import merge_snp_indel_vcf, extract_samples_vcf

wgs_snp_file = f'{source_dir}/Sorghum_d8.noduplicates.allChr.snp._markernamesadded_imputed_snpeff.vcf.gz'
wgs_indel_file = f'{source_dir}/Sorghum_d8.noduplicates.allChr.indel._markernamesadded_imputed_snpeff.vcf.gz'
landraces_metadata_file = f'{source_dir}/landraces_metadata.csv'
landraces_metadata_df = pd.read_csv(landraces_metadata_file)

unique_site_file = f'{results_dir}/Sorghum_d8.allChr.imputed_snpeff.vcf.gz'
if not os.path.exists(unique_site_file):
    merge_snp_indel_vcf(wgs_snp_file, wgs_indel_file, unique_site_file)
else:
    print(f'Reusing merged VCF: {unique_site_file}')

landraces_vcf_file = f'{results_dir}/landraces.vcf.gz'
if not os.path.exists(landraces_vcf_file):
    sample_list = landraces_metadata_df['LIB'].tolist()
    if not pd.Index(sample_list).is_unique:
        raise ValueError('LIB sample IDs must be unique for genotype alignment.')
    extract_samples_vcf(unique_site_file, sample_list, landraces_vcf_file)
else:
    print(f'Reusing landrace VCF: {landraces_vcf_file}')

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:50] Preparing or reusing VCF inputs
Reusing merged VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/Sorghum_d8.allChr.imputed_snpeff.vcf.gz
Reusing landrace VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces.vcf.gz
[06:54:50] Finished in 0.0s


Genome-wide QC excludes highly heterozygous sites because these may reflect mapping errors. All filtering thresholds include their boundary values.

全基因组质控排除高杂合率位点，因为其可能来自比对错误。所有过滤阈值均包含边界值。

In [6]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Preparing or reusing VCF inputs')

from src.utils import have_file

min_maf = 0.05
max_het = 0.05
n_jobs = 8

qc_output_vcf_file = f'{results_dir}/landraces_qc_maf{min_maf}_het{max_het}.vcf.gz'
qc_output_vcf_stat_h5 = f'{qc_output_vcf_file}.var_stat.h5'
if not have_file(qc_output_vcf_file):
    from src.vcf import quality_control_vcf
    quality_control_vcf(
        landraces_vcf_file, qc_output_vcf_file, min_maf=min_maf, max_het=max_het,
        genome_fasta_file=f'{source_dir}/Sbicolor_730_v5.0.fa.gz', n_jobs=n_jobs,
    )
else:
    print(f'Reusing genome QC VCF: {qc_output_vcf_file}')
if not have_file(qc_output_vcf_stat_h5):
    from src.vcf import build_var_stat_table
    build_var_stat_table(qc_output_vcf_file, qc_output_vcf_stat_h5, n_jobs=n_jobs)
else:
    print(f'Reusing variant statistics: {qc_output_vcf_stat_h5}')

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:50] Preparing or reusing VCF inputs
Reusing genome QC VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_qc_maf0.05_het0.05.vcf.gz
Reusing variant statistics: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_qc_maf0.05_het0.05.vcf.gz.var_stat.h5
[06:54:50] Finished in 0.0s


The lower-MAF CDS panel supports coding-consequence and putative LoF analyses. The heterozygosity limit is the same as in genome-wide QC.

较低 MAF 的 CDS 数据用于编码后果和候选 LoF 分析。杂合率限制与全基因组质控相同。

In [7]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Preparing or reusing VCF inputs')

min_maf = 0.01
max_het = 0.05
n_jobs = 8

ref_gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
cds_output_vcf_file = f'{results_dir}/landraces_cds_maf{min_maf}_het{max_het}.vcf.gz'
cds_output_vcf_file_stat_h5 = f'{cds_output_vcf_file}.var_stat.h5'
if not have_file(cds_output_vcf_file):
    from src.vcf import extract_cds_vcf
    extract_cds_vcf(
        landraces_vcf_file, ref_gff_file, cds_output_vcf_file,
        min_maf=min_maf, max_het=max_het,
    )
else:
    print(f'Reusing CDS QC VCF: {cds_output_vcf_file}')
if not have_file(cds_output_vcf_file_stat_h5):
    from src.vcf import build_var_stat_table
    build_var_stat_table(cds_output_vcf_file, cds_output_vcf_file_stat_h5, n_jobs=n_jobs)
else:
    print(f'Reusing variant statistics: {cds_output_vcf_file_stat_h5}')

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:50] Preparing or reusing VCF inputs
Reusing CDS QC VCF: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_cds_maf0.01_het0.05.vcf.gz
Reusing variant statistics: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/landraces_cds_maf0.01_het0.05.vcf.gz.var_stat.h5
[06:54:50] Finished in 0.0s


### 1.2 Annotate genome-wide sites with MAF ≥ 0.05

### 1.2 注释全基因组 MAF ≥ 0.05 的位点

Load the genome-wide site table with variant ID, chromosome, position and MAF. Each ID identifies one row; context annotations are added as boolean columns. Existing annotation columns are reused.

载入包含变异 ID、染色体、位置及 MAF 的全基因组位点表。每个 ID 对应一行，context 注释以布尔列加入。已有注释列直接复用。

In [8]:
from src.contexts import load_site_info_df
import pandas as pd

site_contexts_df_h5 = f'{results_dir}/site_contexts.genome.maf0.05.h5'

if not have_file(site_contexts_df_h5):
    site_contexts_df = load_site_info_df(qc_output_vcf_stat_h5)
    site_contexts_df = site_contexts_df[['ID', 'CHROM', 'POS', 'MAF']]
    save_context_table(site_contexts_df, site_contexts_df_h5)
else:
    site_contexts_df = pd.read_hdf(site_contexts_df_h5, key='df')

if not site_contexts_df['ID'].is_unique:
    raise ValueError('Duplicate IDs would make context matching ambiguous.')
print(f'Table: {site_contexts_df_h5}; shape={site_contexts_df.shape}')

contexts_list = [i for i in site_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'POS', 'MAF']]
print(f" {len(contexts_list)} contexts have already been recorded in site_contexts_df")

site_contexts_df

Table: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/site_contexts.genome.maf0.05.h5; shape=(6452577, 43)
 39 contexts have already been recorded in site_contexts_df


,ID,CHROM,POS,MAF,gene_body,genic_2kb,genic_proximal,genic_15kb,TE,repeat,...,low_recombination,med_recombination,high_recombination,maf_0.05_0.1,maf_0.1_0.2,maf_0.2_0.3,maf_0.3_0.4,maf_0.4_0.5,flowering_time_genes,core_flowering_time_genes
0,SNP_Chr01_20650,Chr01,20650,0.230248,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
1,SNP_Chr01_20658,Chr01,20658,0.229120,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
2,INDEL_Chr01_20702,Chr01,20702,0.230248,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
3,SNP_Chr01_20730,Chr01,20730,0.284424,False,True,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
4,INDEL_Chr01_20736,Chr01,20736,0.178330,False,True,True,True,False,False,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6452572,SNP_Chr10_62843420,Chr10,62843420,0.059819,True,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
6452573,SNP_Chr10_62843678,Chr10,62843678,0.075621,False,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
6452574,SNP_Chr10_62843803,Chr10,62843803,0.059819,False,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False
6452575,SNP_Chr10_62843840,Chr10,62843840,0.058691,False,True,True,True,False,False,...,False,False,False,True,False,False,False,False,False,False


#### Genomic composition and gene structure

Gene neighborhoods use 0, 2, 5, and 15 kb flanks; TE classes are DNA, LINE, LTR, and SINE. `other_intergenic` excludes `genic_2kb` and TE. These contexts can overlap. GFF-derived coordinates are 1-based inclusive; strand-aware structure uses the longest transcript.

#### 基因组组成与基因结构

基因邻域采用 0、2、5、15 kb flanks；TE 类别为 DNA、LINE、LTR 和 SINE。`other_intergenic` 排除 `genic_2kb` 和 TE。这些 contexts 可以重叠。GFF 衍生坐标为 1-based 闭区间；链方向相关结构采用最长转录本。

In [9]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

# genomic composition contexts

genomic_composition_contexts = ['gene_body', 'genic_2kb', 'genic_proximal', "genic_15kb", "TE", "other_intergenic"]
ref_gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
ref_repeat_gff_file = f'{source_dir}/Sbicolor_730_v5.1.repeatmasked_assembly_v5.0.gff3.gz'

if not all([c in site_contexts_df.columns for c in genomic_composition_contexts]):
    from src.contexts import get_genomic_composition_contexts
    site_contexts_df = get_genomic_composition_contexts(site_contexts_df, ref_gff_file, ref_repeat_gff_file)
    save_context_table(site_contexts_df, site_contexts_df_h5)

# gene structure contexts

gene_structure_contexts = ['upstream_2kb', "five_prime_UTR", 'CDS', 'intron', "three_prime_UTR", 'downstream_2kb']
ref_gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'

if not all([c in site_contexts_df.columns for c in gene_structure_contexts]):
    from src.contexts import get_gene_struc_contexts
    site_contexts_df = get_gene_struc_contexts(site_contexts_df, ref_gff_file, gene_flank_size=2000)
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:57] Annotating or reusing saved context columns
[06:54:57] Finished in 0.0s


#### Coding consequences

Read SnpEff `ANN` and assign the highest-priority consequence per ID: HIGH impact as putative LoF, then MODERATE missense, then LOW synonymous. Genome annotations are matched by ID to the MAF ≥ 0.01 CDS panel; these are putative consequences, not experimentally validated effects.

#### 编码后果

读取 SnpEff `ANN`，每个 ID 按优先级分类：HIGH impact 为候选 LoF，其次 MODERATE missense，再次 LOW synonymous。全基因组注释通过 ID 与 MAF ≥ 0.01 CDS 数据匹配；这些为候选后果，并非实验验证效应。

In [10]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

coding_consequence_contexts = ['missense', 'synonymous', 'putative_LoF']
cds_snpeff_vcf_file = cds_output_vcf_file
if not all([c in site_contexts_df.columns for c in coding_consequence_contexts]):
    from src.contexts import get_coding_mutation_contexts
    site_contexts_df = get_coding_mutation_contexts(site_contexts_df, cds_snpeff_vcf_file)
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:57] Annotating or reusing saved context columns
[06:54:57] Finished in 0.0s


#### ACRs

ACR data are from Supplementary Table 12 of Lu et al. (2019), DOI `10.1038/s41477-019-0548-z`, provided as `ACRs.v5.bed.gz`. Membership is determined by whether a variant position falls within an interval; both interval endpoints are included.

#### ACRs

ACR 数据来自 Lu et al. (2019) 的 Supplementary Table 12，DOI `10.1038/s41477-019-0548-z`，输入文件为 `ACRs.v5.bed.gz`。根据变异位置是否落在区间内判断成员关系，区间两端均计入。

In [11]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

acrs_contexts = ['ACR']
acrs_v5_bed_file = f'{source_dir}/ACRs.v5.bed.gz'
if not all([c in site_contexts_df.columns for c in acrs_contexts]):
    from src.contexts import get_bed_contexts
    site_contexts_df = get_bed_contexts(site_contexts_df, acrs_v5_bed_file, context_name='ACR')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:58] Annotating or reusing saved context columns
[06:54:58] Finished in 0.0s


#### UMRs

UMR data are from Dataset_S08, DOI `10.1073/pnas.2010250117`, provided as `UMRs.v5.bed.gz`. Membership uses the same inclusive interval endpoints as ACRs.

#### UMRs

UMR 数据来自 Dataset_S08，DOI `10.1073/pnas.2010250117`，输入文件为 `UMRs.v5.bed.gz`。成员关系采用与 ACR 相同的闭区间端点。

In [12]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

umrs_contexts = ['UMR']
umrs_bed_file = f'{source_dir}/UMRs.v5.bed.gz'
if not all([c in site_contexts_df.columns for c in umrs_contexts]):
    from src.contexts import get_bed_contexts
    site_contexts_df = get_bed_contexts(site_contexts_df, umrs_bed_file, context_name='UMR')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:58] Annotating or reusing saved context columns
[06:54:58] Finished in 0.0s


#### riboSNitch

Keep the minimum `CC_BPPROB` record per variant ID and classify values below 0.8 as riboSNitch. Rows missing any selected input field are excluded.

#### riboSNitch

每个变异 ID 保留最小 `CC_BPPROB` 记录，将低于 0.8 的值分类为 riboSNitch。所选输入字段中任一缺失的行被排除。

In [13]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

ribosnitch_contexts = ['riboSNitch']
snpfold_results_file = f'{source_dir}/snpfold.results.csv.gz'
if not all([c in site_contexts_df.columns for c in ribosnitch_contexts]):
    from src.contexts import get_ID_list_contexts
    snpfold_results_df = pd.read_csv(snpfold_results_file, sep=",", header=0)
    snpfold_results_df = snpfold_results_df[['variant_id', 'chrom', 'pos', 'CC_BPPROB']]
    snpfold_results_df.dropna(inplace=True, ignore_index=True)
    snpfold_results_df.rename(columns={'chrom': 'CHROM', 'pos': 'POS', 'variant_id': 'ID'}, inplace=True)
    # remove duplicates based on 'ID', keeping the smallest 'CC_BPPROB' value for each duplicate
    snpfold_results_df = snpfold_results_df.sort_values(by=['CC_BPPROB'], ascending=True).drop_duplicates(subset=['ID'], keep='first')
    snpfold_results_df.sort_values(by=['CHROM', 'POS'], inplace=True)
    snpfold_results_df['riboSNitch'] = snpfold_results_df.apply(lambda x: "riboSNitch" if x['CC_BPPROB'] < 0.8 else "non-riboSNitch", axis=1)
    riboSNitch_ID_list = snpfold_results_df[snpfold_results_df['riboSNitch'] == 'riboSNitch']['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, riboSNitch_ID_list, context_name='riboSNitch')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:58] Annotating or reusing saved context columns
[06:54:58] Finished in 0.0s


#### Phylogenetic conservation

Within `genic_15kb`, use phyloP > 1, −1 ≤ phyloP ≤ 1, and phyloP < −1 for conserved, neutral, and accelerated sites. `non_CEs` is the remaining neighborhood IDs, including missing conservation information; it is not a demonstrated absence of conservation.

#### 系统发育保守性

在 `genic_15kb` 内，采用 phyloP > 1、−1 ≤ phyloP ≤ 1、phyloP < −1 划分保守、中性及加速位点。`non_CEs` 为剩余邻域 ID，包含缺失保守性信息的位点，并不代表已证明没有保守性。

In [14]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

phyloP_file = f'{source_dir}/Sbi_PhyloP.csv.gz'
phylo_conservation_contexts = ['non_CEs', 'conserved_sites', 'neutral_sites', 'accelerated_sites']
if not all([c in site_contexts_df.columns for c in phylo_conservation_contexts]):
    from src.contexts import get_ID_list_contexts
    phyloP_df = pd.read_csv(phyloP_file, sep=",", header=0)
    genic_sites_ID_list = site_contexts_df[site_contexts_df['genic_15kb'] == True]['ID'].tolist()
    # conserved_sites
    conserved_sites_ID_list = phyloP_df[phyloP_df['phyloP'] > 1]['ID'].tolist()
    conserved_sites_ID_list = set(conserved_sites_ID_list).intersection(set(genic_sites_ID_list))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, conserved_sites_ID_list, context_name='conserved_sites')
    # neutral_sites
    neutral_sites_ID_list = phyloP_df[(phyloP_df['phyloP'] >= -1) & (phyloP_df['phyloP'] <= 1)]['ID'].tolist()
    neutral_sites_ID_list = set(neutral_sites_ID_list).intersection(set(genic_sites_ID_list))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, neutral_sites_ID_list, context_name='neutral_sites')
    # accelerated_sites
    accelerated_sites_ID_list = phyloP_df[phyloP_df['phyloP'] < -1]['ID'].tolist()
    accelerated_sites_ID_list = set(accelerated_sites_ID_list).intersection(set(genic_sites_ID_list))
    site_contexts_df = get_ID_list_contexts(site_contexts_df, accelerated_sites_ID_list, context_name='accelerated_sites')
    # other sites
    non_CEs_ID_list = set(genic_sites_ID_list) - set(conserved_sites_ID_list) - set(neutral_sites_ID_list) - set(accelerated_sites_ID_list)
    site_contexts_df = get_ID_list_contexts(site_contexts_df, non_CEs_ID_list, context_name='non_CEs')

    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:59] Annotating or reusing saved context columns
[06:54:59] Finished in 0.0s


#### Nucleotide diversity in gene neighborhoods

Calculate π from the extracted landrace VCF in merged, chromosome-clipped 15 kb gene neighborhoods, using 500 bp windows with at least three variants. Quartiles are calculated across retained windows: low < Q25, medium Q25 ≤ π < Q75, and high ≥ Q75.

#### 基因邻域核苷酸多样性

由提取后的 landrace VCF 计算 π，采用合并且按染色体边界截断的 15 kb 基因邻域，以及至少含三个变异的 500 bp 窗口。在保留窗口中计算四分位数：低 < Q25、中 Q25 ≤ π < Q75、高 ≥ Q75。

In [15]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

ref_gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
ref_genome_fa = f'{source_dir}/Sbicolor_730_v5.0.fa.gz'
vcf_file = f'{results_dir}/landraces.vcf.gz'
pi_contexts = ['low_pi', 'med_pi', 'high_pi']
window_size = 500
gene_flank_size = 15000
min_snp_count = 3

if not all([c in site_contexts_df.columns for c in pi_contexts]):
    from src.contexts import get_pi_statistics, get_window_contexts
    pi_statistics_h5 = f'{results_dir}/pi_statistics.genic_15kb.b{window_size}.min_snp{min_snp_count}.h5'
    if not have_file(pi_statistics_h5):
        pi_df = get_pi_statistics(
            ref_gff_file, ref_genome_fa, vcf_file, window_size=window_size,
            gene_flank_size=gene_flank_size, min_snp_count=min_snp_count,
        )
        save_context_table(pi_df, pi_statistics_h5)
    else:
        pi_df = pd.read_hdf(pi_statistics_h5, key='df')
        print(f'Reusing window statistics: {pi_statistics_h5}; rows={len(pi_df):,}')
    low_threshold = pi_df['PI'].quantile(0.25)
    high_threshold = pi_df['PI'].quantile(0.75)
    low_pi_df = pi_df[pi_df['PI'] < low_threshold]
    med_pi_df = pi_df[(pi_df['PI'] >= low_threshold) & (pi_df['PI'] < high_threshold)]
    high_pi_df = pi_df[pi_df['PI'] >= high_threshold]
    site_contexts_df = get_window_contexts(site_contexts_df, low_pi_df, 'low_pi')
    site_contexts_df = get_window_contexts(site_contexts_df, med_pi_df, 'med_pi')
    site_contexts_df = get_window_contexts(site_contexts_df, high_pi_df, 'high_pi')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:59] Annotating or reusing saved context columns
[06:54:59] Finished in 0.0s


#### Genome-wide nucleotide diversity

Calculate diversity across the genome in 500 bp windows within 1 Mb processing chunks. Retain windows containing at least three variants, then classify their π values using the same quartile boundaries as the gene-neighborhood analysis.

#### 全基因组核苷酸多样性

在 1 Mb 处理分块内，按 500 bp 窗口计算全基因组多样性。保留至少含三个变异的窗口，再采用与基因邻域分析相同的四分位数边界划分 π。

In [16]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

ref_gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
ref_genome_fa = f'{source_dir}/Sbicolor_730_v5.0.fa.gz'
vcf_file = f'{results_dir}/landraces.vcf.gz'
recombination_file = f'{source_dir}/avg_cM_genomewide_NAM_population.csv.gz'
pi_contexts = ['low_pi_genome', 'med_pi_genome', 'high_pi_genome']
window_size = 500
min_snp_count = 3

if not all([c in site_contexts_df.columns for c in pi_contexts]):
    from src.contexts import get_genome_allele_statistics, get_window_contexts
    pi_statistics_h5 = f'{results_dir}/pi_statistics.genome.b{window_size}.min_snp{min_snp_count}.h5'
    if not have_file(pi_statistics_h5):
        pi_df = get_genome_allele_statistics(
            ref_genome_fa, vcf_file, recombination_file, window_size=window_size,
            min_snp_count=min_snp_count, chunk_size=1_000_000, n_jobs=1,
        )
        save_context_table(pi_df, pi_statistics_h5)
    else:
        pi_df = pd.read_hdf(pi_statistics_h5, key='df')
        print(f'Reusing window statistics: {pi_statistics_h5}; rows={len(pi_df):,}')
    low_threshold = pi_df['PI'].quantile(0.25)
    high_threshold = pi_df['PI'].quantile(0.75)
    low_pi_df = pi_df[pi_df['PI'] < low_threshold]
    med_pi_df = pi_df[(pi_df['PI'] >= low_threshold) & (pi_df['PI'] < high_threshold)]
    high_pi_df = pi_df[pi_df['PI'] >= high_threshold]
    site_contexts_df = get_window_contexts(site_contexts_df, low_pi_df, 'low_pi_genome')
    site_contexts_df = get_window_contexts(site_contexts_df, med_pi_df, 'med_pi_genome')
    site_contexts_df = get_window_contexts(site_contexts_df, high_pi_df, 'high_pi_genome')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:54:59] Annotating or reusing saved context columns
[06:54:59] Finished in 0.0s


#### Recombination rate

Match sites to inclusive intervals `bp` through `bp+999`; convert `meanCmPerBp` to cM/Mb. Use low rates < 3 cM/Mb, medium rates from 3 up to Q75, and high rates ≥ Q75. Q75 is calculated from map rows strictly above 3 cM/Mb; it is not the quartile of all sites.

#### 重组率

将位点匹配到 `bp` 至 `bp+999` 的闭区间，将 `meanCmPerBp` 转为 cM/Mb。低值 < 3 cM/Mb，中值从 3 至 Q75（不含 Q75），高值 ≥ Q75。Q75 在重组率严格大于 3 cM/Mb 的 map 行中计算，并非全部位点的四分位数。

In [17]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

recombination_contexts = ['low_recombination', 'med_recombination', 'high_recombination']
recombination_file = f'{source_dir}/avg_cM_genomewide_NAM_population.csv.gz'

min_cM_per_Mb = 3

if not all([c in site_contexts_df.columns for c in recombination_contexts]):
    from src.contexts import get_site_recombination_contexts
    site_contexts_df = get_site_recombination_contexts(site_contexts_df, recombination_file, min_CmPerMb=min_cM_per_Mb)
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:00] Annotating or reusing saved context columns
[06:55:00] Finished in 0.0s


#### Minor allele frequency

Use left-closed, right-open bins; the final genome bin includes MAF = 0.5.

#### 次要等位基因频率

采用左闭右开分箱；全基因组最后一箱包含 MAF = 0.5。

In [18]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

maf_contexts = ['maf_0.05_0.1', 'maf_0.1_0.2', 'maf_0.2_0.3', 'maf_0.3_0.4', 'maf_0.4_0.5']

if not all([c in site_contexts_df.columns for c in maf_contexts]):
    from src.contexts import get_ID_list_contexts
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.05) & (site_contexts_df['MAF'] < 0.1)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.05_0.1')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.1) & (site_contexts_df['MAF'] < 0.2)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.1_0.2')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.2) & (site_contexts_df['MAF'] < 0.3)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.2_0.3')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.3) & (site_contexts_df['MAF'] < 0.4)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.3_0.4')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.4) & (site_contexts_df['MAF'] <= 0.5)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.4_0.5')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:00] Annotating or reusing saved context columns
[06:55:00] Finished in 0.0s


#### Flowering-time genes

Match catalog genes with 2 kb flanks. Core genes are records whose `Annotation_source` equals `Literature-curated genes`.

#### 开花时间基因

按 catalog 中基因及其 2 kb flanks 匹配位点。核心基因为 `Annotation_source` 等于 `Literature-curated genes` 的记录。

In [19]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

flowering_genes_file = f'{source_dir}/Sorghum_flowering_genes.csv'
flowering_genes_contexts = ['flowering_time_genes', 'core_flowering_time_genes']

if not all([c in site_contexts_df.columns for c in flowering_genes_contexts]):
    from src.contexts import get_gene_contexts
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_IDs = flowering_genes_df['Gene_ID'].to_list()
    core_flowering_gene_IDs = flowering_genes_df.loc[flowering_genes_df['Annotation_source'] == 'Literature-curated genes', 'Gene_ID'].to_list()
    site_contexts_df = get_gene_contexts(site_contexts_df, flowering_gene_IDs, ref_gff_file, 'flowering_time_genes', flank_size=2000)
    site_contexts_df = get_gene_contexts(site_contexts_df, core_flowering_gene_IDs, ref_gff_file, 'core_flowering_time_genes', flank_size=2000)
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:00] Annotating or reusing saved context columns
[06:55:00] Finished in 0.0s


### 1.3 Annotate CDS sites with MAF ≥ 0.01

### 1.3 注释 CDS 中 MAF ≥ 0.01 的位点

Load the separate CDS table and retain its original ID order. The `_cds_0.01` suffix distinguishes coding contexts from the genome panel.

载入独立 CDS 表并保留原 ID 顺序。`_cds_0.01` 后缀区分编码 contexts 与全基因组数据。

In [20]:
from src.contexts import load_site_info_df
import pandas as pd

site_contexts_df_h5 = f'{results_dir}/site_contexts.cds.maf0.01.h5'

if not have_file(site_contexts_df_h5):
    site_contexts_df = load_site_info_df(cds_output_vcf_file_stat_h5)
    site_contexts_df = site_contexts_df[['ID', 'CHROM', 'POS', 'MAF']]
    save_context_table(site_contexts_df, site_contexts_df_h5)
else:
    site_contexts_df = pd.read_hdf(site_contexts_df_h5, key='df')

if not site_contexts_df['ID'].is_unique:
    raise ValueError('Duplicate IDs would make context matching ambiguous.')
print(f'Table: {site_contexts_df_h5}; shape={site_contexts_df.shape}')

contexts_list = [i for i in site_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'POS', 'MAF']]
print(f" {len(contexts_list)} contexts have already been recorded in site_contexts_df")

site_contexts_df

Table: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/site_contexts.cds.maf0.01.h5; shape=(386925, 16)
 12 contexts have already been recorded in site_contexts_df


,ID,CHROM,POS,MAF,missense_cds_0.01,synonymous_cds_0.01,putative_LoF_cds_0.01,maf_0.01_0.03_cds_0.01,maf_0.03_0.05_cds_0.01,maf_0.05_0.1_cds_0.01,maf_0.1_0.2_cds_0.01,maf_0.2_0.3_cds_0.01,maf_0.3_0.4_cds_0.01,maf_0.4_0.5_cds_0.01,flowering_time_genes,core_flowering_time_genes
0,SNP_Chr01_22482,Chr01,22482,0.019187,True,False,False,True,False,False,False,False,False,False,False,False
1,SNP_Chr01_22585,Chr01,22585,0.019187,False,True,False,True,False,False,False,False,False,False,False,False
2,SNP_Chr01_22703,Chr01,22703,0.019187,True,False,False,True,False,False,False,False,False,False,False,False
3,SNP_Chr01_33180,Chr01,33180,0.024831,False,True,False,True,False,False,False,False,False,False,False,False
4,SNP_Chr01_34142,Chr01,34142,0.382619,True,False,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
386920,SNP_Chr10_62837224,Chr10,62837224,0.051919,False,True,False,False,False,True,False,False,False,False,False,False
386921,SNP_Chr10_62841974,Chr10,62841974,0.453725,True,False,False,False,False,False,False,False,False,True,False,False
386922,SNP_Chr10_62842354,Chr10,62842354,0.016930,False,True,False,True,False,False,False,False,False,False,False,False
386923,INDEL_Chr10_62842730,Chr10,62842730,0.030474,False,False,True,False,True,False,False,False,False,False,False,False


#### Coding consequences

Apply the same SnpEff priority rule to the CDS panel.

#### 编码后果

对 CDS 数据采用相同 SnpEff 优先级规则。

In [21]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

coding_consequence_contexts = ['missense_cds_0.01', 'synonymous_cds_0.01', 'putative_LoF_cds_0.01']
cds_snpeff_vcf_file = cds_output_vcf_file
if not all([c in site_contexts_df.columns for c in coding_consequence_contexts]):
    from src.contexts import get_coding_mutation_contexts
    site_contexts_df = get_coding_mutation_contexts(site_contexts_df, cds_snpeff_vcf_file, dataset='_cds_0.01')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:01] Annotating or reusing saved context columns
[06:55:01] Finished in 0.0s


#### Minor allele frequency

Divide CDS variants into seven left-closed, right-open MAF bins. The final CDS bin is 0.4 ≤ MAF < 0.5, whereas the genome-wide final bin includes MAF = 0.5.

#### 次要等位基因频率

将 CDS 变异划分为七个左闭右开的 MAF 分箱。CDS 最后一箱为 0.4 ≤ MAF < 0.5，而全基因组最后一箱包含 MAF = 0.5。

In [22]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

maf_contexts = ['maf_0.01_0.03_cds_0.01', 'maf_0.03_0.05_cds_0.01', 'maf_0.05_0.1_cds_0.01', 'maf_0.1_0.2_cds_0.01', 'maf_0.2_0.3_cds_0.01', 'maf_0.3_0.4_cds_0.01', 'maf_0.4_0.5_cds_0.01']

if not all([c in site_contexts_df.columns for c in maf_contexts]):
    from src.contexts import get_ID_list_contexts
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.01) & (site_contexts_df['MAF'] < 0.03)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.01_0.03_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.03) & (site_contexts_df['MAF'] < 0.05)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.03_0.05_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.05) & (site_contexts_df['MAF'] < 0.1)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.05_0.1_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.1) & (site_contexts_df['MAF'] < 0.2)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.1_0.2_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.2) & (site_contexts_df['MAF'] < 0.3)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.2_0.3_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.3) & (site_contexts_df['MAF'] < 0.4)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.3_0.4_cds_0.01')
    used_id_list = site_contexts_df[(site_contexts_df['MAF'] >= 0.4) & (site_contexts_df['MAF'] <= 0.5)]['ID'].tolist()
    site_contexts_df = get_ID_list_contexts(site_contexts_df, used_id_list, 'maf_0.4_0.5_cds_0.01')
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:02] Annotating or reusing saved context columns
[06:55:02] Finished in 0.0s


#### Flowering-time genes

Reuse the same catalog and 2 kb flanks as the genome panel.

#### 开花时间基因

采用与全基因组数据相同的 catalog 和 2 kb flanks。

In [23]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

flowering_genes_file = f'{source_dir}/Sorghum_flowering_genes.csv'
flowering_genes_contexts = ['flowering_time_genes', 'core_flowering_time_genes']

if not all([c in site_contexts_df.columns for c in flowering_genes_contexts]):
    from src.contexts import get_gene_contexts
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_IDs = flowering_genes_df['Gene_ID'].to_list()
    core_flowering_gene_IDs = flowering_genes_df.loc[flowering_genes_df['Annotation_source'] == 'Literature-curated genes', 'Gene_ID'].to_list()
    site_contexts_df = get_gene_contexts(site_contexts_df, flowering_gene_IDs, ref_gff_file, 'flowering_time_genes', flank_size=2000)
    site_contexts_df = get_gene_contexts(site_contexts_df, core_flowering_gene_IDs, ref_gff_file, 'core_flowering_time_genes', flank_size=2000)
    save_context_table(site_contexts_df, site_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:02] Annotating or reusing saved context columns
[06:55:02] Finished in 0.0s


## 2. Define gene-level contexts

## 2. 定义基因级 contexts

Load GFF gene records whose chromosome names start with `Chr`, preserving their input order. Gene coordinates are 1-based inclusive.

载入染色体名称以 `Chr` 开头的 GFF gene 记录，保留输入顺序。基因坐标为 1-based 闭区间。

In [24]:
import pandas as pd
from src.utils import have_file

ref_gff_file = f'{source_dir}/Sbicolor_730_v5.1.gene_exons.gff3.gz'
gene_contexts_df_h5 = f'{results_dir}/gene_contexts.h5'

if not have_file(gene_contexts_df_h5):
    from src.contexts import load_gene_info_df
    gene_contexts_df = load_gene_info_df(ref_gff_file)
    save_context_table(gene_contexts_df, gene_contexts_df_h5)
else:
    gene_contexts_df = pd.read_hdf(gene_contexts_df_h5, key='df')

if not gene_contexts_df['ID'].is_unique:
    raise ValueError('Duplicate IDs would make context matching ambiguous.')
print(f'Table: {gene_contexts_df_h5}; shape={gene_contexts_df.shape}')

contexts_list = [i for i in gene_contexts_df.columns.tolist() if i not in ['ID', 'CHROM', 'START', 'END', 'STRAND']]
print(f" {len(contexts_list)} contexts have already been recorded in gene_contexts_df")

gene_contexts_df

Table: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/gene_contexts.h5; shape=(31819, 18)
 13 contexts have already been recorded in gene_contexts_df


,ID,CHROM,START,END,STRAND,flowering_time_genes,no_expression_genes,high_variance_genes,stably_expressed_genes,module_hub_genes,module_genes,non_module_genes,orphan_genes,single_copy_genes,multi_copy_genes,low_recombination_genes,med_recombination_genes,high_recombination_genes
0,Sobic.001G000100.v5.1,Chr01,22085,23338,+,False,False,False,False,False,True,False,False,True,False,True,False,False
1,Sobic.001G000200.v5.1,Chr01,31590,35375,-,False,False,False,False,True,False,False,False,True,False,True,False,False
2,Sobic.001G000300.v5.1,Chr01,43890,44492,-,False,False,False,False,False,True,False,False,False,True,True,False,False
3,Sobic.001G000400.v5.1,Chr01,44828,63076,-,False,False,False,False,False,True,False,False,True,False,True,False,False
4,Sobic.001G000700.v5.1,Chr01,80708,83931,+,False,False,False,False,False,True,False,False,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31814,Sobic.010G279800.v5.1,Chr10,62816416,62825451,+,False,False,False,False,False,True,False,False,True,False,False,False,False
31815,Sobic.010G279900.v5.1,Chr10,62825908,62834521,-,False,False,False,False,False,True,False,False,True,False,False,False,False
31816,Sobic.010G280000.v5.1,Chr10,62836480,62837370,-,False,False,False,False,False,True,False,False,True,False,False,False,False
31817,Sobic.010G280100.v5.1,Chr10,62838030,62841326,+,False,False,False,False,False,True,False,False,True,False,False,False,False


#### Flowering-time genes

Match the full catalog directly by gene ID.

#### 开花时间基因

通过基因 ID 直接匹配完整 catalog。

In [25]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

flowering_gene_contexts = ['flowering_time_genes']

if not all([c in gene_contexts_df.columns for c in flowering_gene_contexts]):
    from src.contexts import get_ID_list_contexts
    flowering_genes_file = f'{source_dir}/Sorghum_flowering_genes.csv'
    flowering_genes_df = pd.read_csv(flowering_genes_file, sep=",", header=0)
    flowering_gene_IDs = flowering_genes_df['Gene_ID'].to_list()
    gene_contexts_df = get_ID_list_contexts(gene_contexts_df, flowering_gene_IDs, context_name='flowering_time_genes')
    save_context_table(gene_contexts_df, gene_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:03] Annotating or reusing saved context columns
[06:55:03] Finished in 0.0s


#### Gene expression

Set TPM ≤ 2 to zero. Fit LOWESS (fraction 0.1) to log2(mean+1) versus log2(variance+1); high-variance genes have standardized residual > 1. Stable genes have residual < 1 and retained expression in every sample. Genes absent from the TPM matrix are not classified as no-expression genes by this implementation.

#### 基因表达

将 TPM ≤ 2 设为零。对 log2(mean+1) 与 log2(variance+1) 拟合 LOWESS（fraction 0.1）；高变异基因的标准化残差 > 1。稳定基因满足残差 < 1 且每个样本均有保留表达。当前实现不将 TPM 矩阵中缺失的基因分类为无表达。

In [26]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

expression_gene_contexts = ['no_expression_genes', 'high_variance_genes', 'stably_expressed_genes']

min_expr = 2.0
high_var_threshold = 1.0

if not all([c in gene_contexts_df.columns for c in expression_gene_contexts]):
    from src.contexts import get_expression_contexts
    TPMs_csv_file = f'{source_dir}/Sbi_NCBI_SRA_TPMs.csv.gz'
    gene_contexts_df = get_expression_contexts(
        gene_contexts_df, TPMs_csv_file, min_expr=min_expr, high_var_threshold=high_var_threshold,
    )
    save_context_table(gene_contexts_df, gene_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:03] Annotating or reusing saved context columns
[06:55:03] Finished in 0.0s


#### Co-expression networks

Retain edges with weight ≥ 0.1. In modules with at least five graph vertices, hubs have degree > 2 and at least two centrality z-scores > 1. `module_genes` excludes hubs; `non_module_genes` excludes all module members.

#### 共表达网络

保留 weight ≥ 0.1 的边。在图顶点数至少五的模块中，hub 满足 degree > 2 且至少两个中心性 z-score > 1。`module_genes` 排除 hubs；`non_module_genes` 排除所有模块成员。

In [27]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

GCNs_gene_contexts = ['non_module_genes', 'module_genes', 'module_hub_genes']

edge_cutoff = 0.1

if not all([c in gene_contexts_df.columns for c in GCNs_gene_contexts]):
    from src.contexts import get_GCNs_contexts
    module_file = f'{source_dir}/edge_weight_N3.0.01.clusterone.txt.gz'
    edge_weight_file = f'{source_dir}/edge_weight_N3.0.01.tsv.gz'
    gene_contexts_df = get_GCNs_contexts(gene_contexts_df, edge_weight_file, module_file, cut_off=edge_cutoff)
    save_context_table(gene_contexts_df, gene_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:03] Annotating or reusing saved context columns
[06:55:03] Finished in 0.0s


#### Gene families

Use the `Sbic` field of the Poaceae HOG table. Multiple-copy genes occur in groups with more than one listed sorghum gene; single-copy genes are other listed genes; orphan genes are absent from these assignments.

#### 基因家族

采用 Poaceae HOG 表的 `Sbic` 字段。多拷贝基因为含多个已列高粱基因的组中成员；单拷贝为其余已列基因；孤儿基因未出现在这些分配中。

In [28]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

gene_family_contexts = ['orphan_genes', 'single_copy_genes', 'multi_copy_genes']

if not all([c in gene_contexts_df.columns for c in gene_family_contexts]):
    from src.contexts import get_gene_family_contexts
    gene_family_file = f'{source_dir}/Poaceae_HOG.tsv.gz'
    gene_contexts_df = get_gene_family_contexts(gene_contexts_df, gene_family_file)
    save_context_table(gene_contexts_df, gene_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:04] Annotating or reusing saved context columns
[06:55:04] Finished in 0.0s


#### Gene recombination

Average rates of overlapping 1 kb map intervals without overlap-length weighting. Define quartiles among mapped genes; unmapped genes remain false in all three categories.

#### 基因重组率

对重叠的 1 kb map 区间重组率取均值，不按重叠长度加权。在成功匹配的基因中划分四分位数；未匹配基因在三个类别中均为 false。

In [29]:
stage_start = perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Annotating or reusing saved context columns')

recombination_contexts = ['low_recombination_genes', 'med_recombination_genes', 'high_recombination_genes']

if not all([c in gene_contexts_df.columns for c in recombination_contexts]):
    from src.contexts import get_gene_recombination_contexts
    recombination_file = f'{source_dir}/avg_cM_genomewide_NAM_population.csv.gz'
    gene_contexts_df = get_gene_recombination_contexts(gene_contexts_df, recombination_file)
    save_context_table(gene_contexts_df, gene_contexts_df_h5)

print(f'[{datetime.now():%H:%M:%S}] Finished in {perf_counter() - stage_start:.1f}s')

[06:55:04] Annotating or reusing saved context columns
[06:55:04] Finished in 0.0s


## 3. Summarize genomic contexts

Summarize the contexts listed in `config/contexts_settings.json` for each dataset. Export their counts and fractions as `.csv.gz`. The denominator is the total number of sites or genes in the corresponding table. Contexts can overlap, so their fractions need not sum to one.

## 3. 汇总基因组 contexts

按数据集汇总 `config/contexts_settings.json` 中列出的 contexts，将数量和比例导出为 `.csv.gz`。分母为对应表中的位点或基因总数。Contexts 可以重叠，其比例不必合计为一。

In [30]:
import json

datasets = {
    'genome': f'{results_dir}/site_contexts.genome.maf0.05.h5', # All variants with MAF >= 0.05
    'cds': f'{results_dir}/site_contexts.cds.maf0.01.h5', # All variants in coding regions with MAF >= 0.01
    'gene': f'{results_dir}/gene_contexts.h5' # All genes
}

contexts_settings_json_file = f'{project_dir}/config/contexts_settings.json'
with open(contexts_settings_json_file, "r") as f:
    contexts_settings = json.load(f)

### 3.1 Genome sites with MAF ≥ 0.05

### 3.1 全基因组 MAF ≥ 0.05 的位点

In [31]:
dataset = 'genome'
context_df = pd.read_hdf(datasets[dataset], key='df')
context_ids = list(contexts_settings[dataset])
summary_df = context_df[context_ids].sum().rename('count').to_frame()
summary_df.index.name = 'context'
summary_df['count'] = summary_df['count'].astype(int)
summary_df['total'] = len(context_df)
summary_df['fraction'] = summary_df['count'] / len(context_df)
summary_file = f'{results_dir}/contexts.{dataset}.summary.csv.gz'
summary_df.reset_index().to_csv(summary_file, index=False, compression='gzip')
print(f'Summary: {summary_file}')
print(f'Rows: {len(context_df):,}')
summary_df

Summary: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/contexts.genome.summary.csv.gz
Rows: 6,452,577


,count,total,fraction
context,,,
gene_body,1013897,6452577,0.157131
genic_2kb,2277703,6452577,0.352991
genic_proximal,3077170,6452577,0.476890
genic_15kb,3938903,6452577,0.610439
TE,3736513,6452577,0.579073
other_intergenic,1054970,6452577,0.163496
upstream_2kb,888419,6452577,0.137684
five_prime_UTR,87769,6452577,0.013602
CDS,240922,6452577,0.037337


### 3.2 CDS sites with MAF ≥ 0.01

### 3.2 CDS 中 MAF ≥ 0.01 的位点

In [32]:
dataset = 'cds'
context_df = pd.read_hdf(datasets[dataset], key='df')
context_ids = list(contexts_settings[dataset])
summary_df = context_df[context_ids].sum().rename('count').to_frame()
summary_df.index.name = 'context'
summary_df['count'] = summary_df['count'].astype(int)
summary_df['total'] = len(context_df)
summary_df['fraction'] = summary_df['count'] / len(context_df)
summary_file = f'{results_dir}/contexts.{dataset}.summary.csv.gz'
summary_df.reset_index().to_csv(summary_file, index=False, compression='gzip')
print(f'Summary: {summary_file}')
print(f'Rows: {len(context_df):,}')
summary_df

Summary: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/contexts.cds.summary.csv.gz
Rows: 386,925


,count,total,fraction
context,,,
missense_cds_0.01,186477,386925,0.481946
synonymous_cds_0.01,174908,386925,0.452046
putative_LoF_cds_0.01,12820,386925,0.033133
maf_0.01_0.03_cds_0.01,107212,386925,0.277087
maf_0.03_0.05_cds_0.01,38791,386925,0.100255
maf_0.05_0.1_cds_0.01,56971,386925,0.147240
maf_0.1_0.2_cds_0.01,70340,386925,0.181792
maf_0.2_0.3_cds_0.01,44412,386925,0.114782
maf_0.3_0.4_cds_0.01,35514,386925,0.091785


### 3.3 Gene-level contexts

### 3.3 基因级 contexts

In [33]:
dataset = 'gene'
context_df = pd.read_hdf(datasets[dataset], key='df')
context_ids = list(contexts_settings[dataset])
summary_df = context_df[context_ids].sum().rename('count').to_frame()
summary_df.index.name = 'context'
summary_df['count'] = summary_df['count'].astype(int)
summary_df['total'] = len(context_df)
summary_df['fraction'] = summary_df['count'] / len(context_df)
summary_file = f'{results_dir}/contexts.{dataset}.summary.csv.gz'
summary_df.reset_index().to_csv(summary_file, index=False, compression='gzip')
print(f'Summary: {summary_file}')
print(f'Rows: {len(context_df):,}')
summary_df

Summary: /lustre/home/dobby/Work/Project/functional-genomic-local-adaptation-sorghum/data/results/contexts.gene.summary.csv.gz
Rows: 31,819


,count,total,fraction
context,,,
flowering_time_genes,312,31819,0.009805
no_expression_genes,1232,31819,0.038719
high_variance_genes,5195,31819,0.163267
stably_expressed_genes,1350,31819,0.042427
non_module_genes,4267,31819,0.134102
module_genes,20419,31819,0.641723
module_hub_genes,7133,31819,0.224174
orphan_genes,3237,31819,0.101732
multi_copy_genes,12421,31819,0.390364
